# Предсказание цены дома: полный цикл обучения модели

**Задача:** по характеристикам дома (площадь, качество, год постройки, район и т.д.) предсказать его **цену продажи**.

Цена — это непрерывное число, поэтому это задача **регрессии**.

**Датасет:** *Ames Housing* (город Эймс, штат Айова, США), 1460 домов и 80 признаков. Загружается напрямую через `sklearn.datasets.fetch_openml`.

### План ноутбука
1. Импорт библиотек
2. Загрузка данных
3. Разведочный анализ данных (EDA)
4. Подготовка данных: выбор признаков, перевод в м², удаление выбросов
5. Разделение на train / test
6. Препроцессинг через `Pipeline` и `ColumnTransformer`
7. Базовая модель (baseline)
8. Простая линейная регрессия «цена ~ площадь»
9. Сравнение нескольких моделей через кросс-валидацию
10. Подбор гиперпараметров (`GridSearchCV`)
11. Финальная оценка на тестовой выборке
12. Важность признаков
13. Сохранение и загрузка модели, предсказание для нового дома
14. Бонус: та же задача как **классификация** (дешёвый / средний / дорогой)

## 1. Импорт библиотек

- `pandas`, `numpy` — работа с таблицами и числами
- `matplotlib` — графики
- `sklearn` — всё для машинного обучения: модели, препроцессинг, метрики, валидация
- `joblib` — сохранение обученной модели в файл

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_validate, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42          # фиксируем случайность, чтобы результаты воспроизводились
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (8, 5)

## 2. Загрузка данных

`fetch_openml` скачивает датасет с сайта OpenML (нужен интернет при первом запуске, потом он кэшируется).

- `as_frame=True` — получаем данные сразу как `pandas.DataFrame`
- `data.frame` — вся таблица (признаки + целевая колонка `SalePrice`)

In [ ]:
data = fetch_openml(name="house_prices", as_frame=True)
df = data.frame.copy()

print("Размер таблицы:", df.shape)
df.head()

## 3. Разведочный анализ данных (EDA)

Прежде чем обучать модель, нужно понять данные:
- какие есть колонки и их типы;
- есть ли пропуски;
- как распределена целевая переменная;
- какие признаки сильнее всего связаны с ценой.

### 3.1 Типы данных и пропуски

In [ ]:
print("Числовых колонок:   ", df.select_dtypes(include="number").shape[1])
print("Категориальных колонок:", df.select_dtypes(exclude="number").shape[1])

missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print("\nДоля пропусков по колонкам (топ-15):")
print((missing.head(15) * 100).round(1).astype(str) + " %")

Некоторые колонки (`PoolQC`, `MiscFeature`, `Alley`) почти полностью пустые. Обычно это значит «этого нет» (например, нет бассейна). Мы такие колонки использовать не будем.

### 3.2 Целевая переменная `SalePrice`

Посмотрим на распределение цены.

In [ ]:
print(df["SalePrice"].describe().round(0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df["SalePrice"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Цена продажи ($)")
axes[1].hist(np.log1p(df["SalePrice"]), bins=50, color="seagreen", edgecolor="white")
axes[1].set_title("log(1 + цена)")
plt.show()

**Вывод:** распределение цены **скошено вправо**: много домов средней цены и мало очень дорогих.

Если взять логарифм, распределение становится почти нормальным. Это помогает линейным моделям, и ошибка становится «относительной»: промах на 10 % для дешёвого и дорогого дома штрафуется одинаково. Поэтому дальше мы будем обучать модели на **log(цены)**, а предсказания переводить обратно в доллары.

### 3.3 Связь цены и площади

`GrLivArea` — жилая площадь над землёй в **квадратных футах** (1 фут² = 0.0929 м²).

In [ ]:
plt.scatter(df["GrLivArea"] * 0.0929, df["SalePrice"], alpha=0.4, s=12)
plt.xlabel("Жилая площадь, м²")
plt.ylabel("Цена, $")
plt.title("Цена vs площадь")
plt.show()

**Вывод:** связь явно положительная: чем больше площадь, тем выше цена.

Справа внизу видны **выбросы**: несколько огромных домов (> 370 м²) с подозрительно низкой ценой. Автор датасета (Dean De Cock) сам рекомендует их удалить, это нетипичные частичные продажи. Удалим их на шаге 4.

### 3.4 Корреляция числовых признаков с ценой

In [ ]:
corr = df.select_dtypes(include="number").corr()["SalePrice"].drop("SalePrice")
top_corr = corr.abs().sort_values(ascending=False).head(12)

corr[top_corr.index].sort_values().plot(kind="barh", color="indianred")
plt.title("Корреляция с SalePrice (топ-12)")
plt.xlabel("Коэффициент корреляции Пирсона")
plt.show()

**Самые сильные признаки:**
- `OverallQual` — общее качество дома (1–10)
- `GrLivArea` — жилая площадь
- `GarageCars` — вместимость гаража
- `TotalBsmtSF` — площадь подвала
- `YearBuilt` — год постройки

На их основе и соберём модель.

### 3.5 Цена по районам (категориальный признак)

In [ ]:
order = df.groupby("Neighborhood", observed=True)["SalePrice"].median().sort_values()
df.boxplot(column="SalePrice", by="Neighborhood", positions=None, rot=90, figsize=(14, 5),
           grid=False)
plt.suptitle("")
plt.title("Цена по районам")
plt.ylabel("Цена, $")
plt.show()
print("Самый дешёвый район (медиана):", order.index[0], int(order.iloc[0]))
print("Самый дорогой район (медиана):", order.index[-1], int(order.iloc[-1]))

**Вывод:** район сильно влияет на цену, медиана отличается в несколько раз. Это категориальный признак, поэтому его нужно будет закодировать (One-Hot Encoding).

## 4. Подготовка данных

### 4.1 Удаление выбросов
Удаляем дома с площадью > 4000 фут² и ценой < 300 000 $ (те самые точки справа внизу).

In [ ]:
outliers = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300_000)
print("Удаляем выбросов:", outliers.sum())
df = df[~outliers].reset_index(drop=True)

### 4.2 Выбор признаков и перевод площадей в м²

Мы берём небольшой понятный набор признаков: так модель проще объяснить, а качество получается почти такое же, как на всех 80 колонках.

| Признак | Описание |
|---|---|
| `LivingArea_m2` | жилая площадь, м² |
| `LotArea_m2` | площадь участка, м² |
| `BsmtArea_m2` | площадь подвала, м² |
| `OverallQual` | общее качество (1–10) |
| `YearBuilt` | год постройки |
| `GarageCars` | мест в гараже |
| `FullBath` | полноценных ванных |
| `TotRmsAbvGrd` | всего комнат |
| `Neighborhood` | район (категория) |
| `KitchenQual` | качество кухни (категория) |

In [ ]:
SQFT_TO_M2 = 0.0929

df["LivingArea_m2"] = df["GrLivArea"]   * SQFT_TO_M2
df["LotArea_m2"]    = df["LotArea"]     * SQFT_TO_M2
df["BsmtArea_m2"]   = df["TotalBsmtSF"] * SQFT_TO_M2

num_features = ["LivingArea_m2", "LotArea_m2", "BsmtArea_m2", "OverallQual",
                "YearBuilt", "GarageCars", "FullBath", "TotRmsAbvGrd"]
cat_features = ["Neighborhood", "KitchenQual"]
features = num_features + cat_features
target = "SalePrice"

# категориальные колонки приводим к обычным строкам (object), чтобы sklearn их корректно обработал
for c in cat_features:
    df[c] = df[c].astype("object")

X = df[features]
y = df[target].astype(float)

print("X:", X.shape, " y:", y.shape)
X.head()

## 5. Разделение на обучающую и тестовую выборки

- **train (80 %)** — на нём обучаем модели и подбираем параметры;
- **test (20 %)** — откладываем и **не трогаем** до самого конца. Это честная проверка того, как модель работает на новых, невиданных данных.

`random_state` фиксирует разбиение, чтобы результаты повторялись.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 6. Препроцессинг: `Pipeline` + `ColumnTransformer`

Разным типам колонок нужна разная обработка:

**Числовые признаки:**
1. `SimpleImputer(strategy="median")` — заполняет пропуски медианой;
2. `StandardScaler()` — приводит к среднему 0 и стандартному отклонению 1. Это важно для линейных моделей, деревьям не мешает.

**Категориальные признаки:**
1. `SimpleImputer(strategy="most_frequent")` — заполняет пропуски самым частым значением;
2. `OneHotEncoder` — превращает категорию в набор колонок 0/1. `handle_unknown="ignore"` нужен, чтобы модель не падала, если в тесте встретится новый район.

**Зачем Pipeline?** Все шаги «обучаются» только на train (медианы, средние, список категорий), а потом применяются к test. Так нет **утечки данных** (data leakage): информация из теста не попадает в обучение.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe,     num_features),
    ("cat", categorical_pipe, cat_features),
])

### Обучение на log(цены)

`TransformedTargetRegressor` автоматически:
- перед обучением применяет `log1p` к `y`;
- после предсказания применяет обратную функцию `expm1`, и мы снова получаем доллары.

Напишем функцию, которая собирает полный пайплайн «препроцессинг → модель»:

In [ ]:
def make_model(regressor):
    pipe = Pipeline([
        ("prep",  preprocessor),
        ("model", regressor),
    ])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)


def evaluate(y_true, y_pred, name="model"):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{name:<25} MAE = {mae:>9,.0f} $   RMSE = {rmse:>9,.0f} $   R² = {r2:.3f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

**Метрики:**
- **MAE** (средняя абсолютная ошибка): насколько долларов модель ошибается в среднем. Легко интерпретировать.
- **RMSE** (корень из средней квадратичной ошибки): сильнее штрафует большие промахи.
- **R²** (коэффициент детерминации): какую долю разброса цен объясняет модель. 1.0 — идеально, 0 — не лучше, чем просто предсказывать среднее.

## 7. Базовая модель (baseline)

Самая глупая модель всегда предсказывает **медианную цену** из train. Любая нормальная модель обязана быть лучше. Если нет, значит, что-то сломано.

In [ ]:
baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
evaluate(y_test, baseline.predict(X_test), "Baseline (медиана)");

## 8. Простая линейная регрессия: цена ~ площадь

Начнём с самой простой модели с одним признаком. Модель ищет прямую:

$$\text{цена} = w \cdot \text{площадь} + b$$

In [ ]:
lin_simple = LinearRegression().fit(X_train[["LivingArea_m2"]], y_train)
pred_simple = lin_simple.predict(X_test[["LivingArea_m2"]])

print(f"Наклон w = {lin_simple.coef_[0]:,.0f} $ за 1 м²")
print(f"Свободный член b = {lin_simple.intercept_:,.0f} $")
evaluate(y_test, pred_simple, "LinReg (только площадь)")

xs = np.linspace(X["LivingArea_m2"].min(), X["LivingArea_m2"].max(), 100).reshape(-1, 1)
plt.scatter(X_test["LivingArea_m2"], y_test, alpha=0.4, s=12, label="Тест: реальные цены")
plt.plot(xs, lin_simple.predict(pd.DataFrame(xs, columns=["LivingArea_m2"])),
         color="red", lw=2, label="Линия регрессии")
plt.xlabel("Жилая площадь, м²"); plt.ylabel("Цена, $"); plt.legend()
plt.title("Линейная регрессия: цена ~ площадь")
plt.show()

**Интерпретация:** каждый дополнительный м² добавляет к цене примерно `w` долларов.

R² около 0.5 означает, что одна площадь объясняет примерно половину разброса цен. Остальное зависит от качества, района, года и т.д. Добавим остальные признаки.

## 9. Сравнение моделей через кросс-валидацию

**Кросс-валидация (K-Fold, K=5):** train делится на 5 частей. Модель 5 раз обучается на 4 частях и проверяется на оставшейся. Так мы получаем 5 оценок и усредняем их. Это надёжнее, чем одна проверка, и **тест при этом мы не трогаем**.

Сравниваем 4 модели:
- **LinearRegression** — обычная линейная регрессия;
- **Ridge** — линейная регрессия с L2-регуляризацией (штраф за большие веса, меньше переобучения);
- **RandomForest** — ансамбль из многих решающих деревьев, усредняет их ответы;
- **GradientBoosting** — деревья строятся последовательно, каждое исправляет ошибки предыдущих.

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge":            Ridge(alpha=1.0),
    "RandomForest":     RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

for name, reg in models.items():
    scores = cross_validate(
        make_model(reg), X_train, y_train, cv=cv,
        scoring={"MAE": "neg_mean_absolute_error",
                 "RMSE": "neg_root_mean_squared_error",
                 "R2": "r2"},
    )
    cv_results.append({
        "model": name,
        "MAE":  -scores["test_MAE"].mean(),
        "RMSE": -scores["test_RMSE"].mean(),
        "R2":    scores["test_R2"].mean(),
        "R2_std": scores["test_R2"].std(),
    })

cv_df = pd.DataFrame(cv_results).sort_values("RMSE").reset_index(drop=True)
cv_df.style.format({"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.3f}", "R2_std": "{:.3f}"})

In [ ]:
cv_df.plot(x="model", y="RMSE", kind="bar", legend=False, color="slateblue", rot=0)
plt.ylabel("RMSE, $ (меньше — лучше)")
plt.title("Кросс-валидация: сравнение моделей")
plt.show()

**Вывод:** все модели намного лучше baseline и регрессии на одной площади.

Обычно лучший результат дают ансамбли деревьев (GradientBoosting / RandomForest): они сами находят нелинейные зависимости и взаимодействия признаков. Ridge близко к ним, при этом он проще и легче интерпретируется.

`R2_std` показывает стабильность модели: чем меньше, тем стабильнее она между фолдами.

## 10. Подбор гиперпараметров (`GridSearchCV`)

**Гиперпараметры** — это настройки модели, которые задаём мы, а не модель учит сама. Например, число деревьев или скорость обучения.

`GridSearchCV` перебирает все комбинации из сетки и для каждой делает кросс-валидацию, а затем выбирает лучшую.

Тюним **GradientBoosting**:
- `n_estimators` — сколько деревьев строить;
- `learning_rate` — насколько сильно каждое дерево корректирует предыдущие;
- `max_depth` — глубина каждого дерева (сложность).

Имена параметров длинные (`regressor__model__...`), потому что модель вложена в `TransformedTargetRegressor`, а затем в `Pipeline`.

In [ ]:
param_grid = {
    "regressor__model__n_estimators":  [200, 400],
    "regressor__model__learning_rate": [0.05, 0.1],
    "regressor__model__max_depth":     [2, 3, 4],
}

grid = GridSearchCV(
    make_model(GradientBoostingRegressor(random_state=RANDOM_STATE)),
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Лучшие параметры:")
for k, v in grid.best_params_.items():
    print(f"  {k.split('__')[-1]:<15} = {v}")
print(f"Лучший RMSE на CV: {-grid.best_score_:,.0f} $")

best_model = grid.best_estimator_     # уже переобучена на всём train с лучшими параметрами

## 11. Финальная оценка на тестовой выборке

Только теперь впервые используем test. Это честная оценка того, как модель будет работать на новых домах.

In [ ]:
y_pred = best_model.predict(X_test)

final = [
    evaluate(y_test, baseline.predict(X_test), "Baseline (медиана)"),
    evaluate(y_test, pred_simple,              "LinReg (только площадь)"),
    evaluate(y_test, y_pred,                   "GradientBoosting (tuned)"),
]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Предсказание vs реальность
axes[0].scatter(y_test, y_pred, alpha=0.5, s=14)
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, "r--", lw=2, label="Идеальное предсказание")
axes[0].set_xlabel("Реальная цена, $"); axes[0].set_ylabel("Предсказанная цена, $")
axes[0].set_title("Предсказание vs реальность"); axes[0].legend()

# Остатки (ошибки)
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, s=14)
axes[1].axhline(0, color="red", ls="--")
axes[1].set_xlabel("Предсказанная цена, $"); axes[1].set_ylabel("Ошибка (реальная − предсказанная), $")
axes[1].set_title("График остатков")
plt.show()

pct_err = (np.abs(residuals) / y_test * 100)
print(f"Медианная ошибка: {pct_err.median():.1f} % от цены дома")
print(f"В пределах ±10 %: {(pct_err <= 10).mean()*100:.0f} % домов")

**Как читать графики:**
- **Слева:** чем ближе точки к красной диагонали, тем точнее модель.
- **Справа:** остатки должны быть хаотично разбросаны вокруг нуля. Если виден «веер» (ошибка растёт с ценой), модель хуже работает с дорогими домами. Это нормально, потому что дорогих домов в данных мало.

## 12. Важность признаков

**Permutation importance:** берём один признак и случайно перемешиваем его значения в тесте. Если качество модели сильно падает, признак важен. Если почти не меняется, модель на него не опирается.

Этот метод работает для любой модели и считается по **исходным** признакам, а не по one-hot колонкам.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                              random_state=RANDOM_STATE, scoring="r2", n_jobs=-1)

imp = pd.Series(perm.importances_mean, index=features).sort_values()
imp.plot(kind="barh", xerr=perm.importances_std[np.argsort(perm.importances_mean)], color="darkorange")
plt.xlabel("Падение R² при перемешивании признака")
plt.title("Важность признаков (permutation importance)")
plt.show()

**Вывод:** обычно самые важные признаки — **качество дома** (`OverallQual`) и **жилая площадь** (`LivingArea_m2`), затем район, площадь подвала и год постройки.

Это совпадает с интуицией и с корреляциями из EDA. Значит, модель учится осмысленным зависимостям.

## 13. Сохранение модели и предсказание для нового дома

`joblib.dump` сохраняет **весь пайплайн целиком**: импутеры, скейлер, энкодер, модель и лог-преобразование. После загрузки модели можно сразу передавать «сырые» данные.

In [ ]:
joblib.dump(best_model, "house_price_model.pkl")
print("Модель сохранена в house_price_model.pkl")

loaded = joblib.load("house_price_model.pkl")

new_house = pd.DataFrame([{
    "LivingArea_m2": 150,
    "LotArea_m2":    800,
    "BsmtArea_m2":   90,
    "OverallQual":   7,
    "YearBuilt":     2005,
    "GarageCars":    2,
    "FullBath":      2,
    "TotRmsAbvGrd":  7,
    "Neighborhood":  "CollgCr",
    "KitchenQual":   "Gd",
}])

price = loaded.predict(new_house)[0]
print(f"Предсказанная цена нового дома: {price:,.0f} $")

## 14. Бонус: та же задача как классификация

Если нужна не точная цена, а **категория** (дешёвый / средний / дорогой), это уже **классификация**.

Порядок действий:
1. Делим цену на 3 равные по количеству домов группы через `pd.qcut`. Границы считаем **по train**, чтобы не было утечки.
2. Обучаем `RandomForestClassifier` с тем же препроцессором.
3. Оцениваем через accuracy, classification report и confusion matrix.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

labels = ["дешёвый", "средний", "дорогой"]
_, bins = pd.qcut(y_train, q=3, retbins=True)
bins[0], bins[-1] = -np.inf, np.inf                      # чтобы в тест попали любые цены

y_train_cls = pd.cut(y_train, bins=bins, labels=labels)
y_test_cls  = pd.cut(y_test,  bins=bins, labels=labels)
print("Границы классов ($):", [f"{b:,.0f}" for b in bins[1:-1]])

clf = Pipeline([
    ("prep",  preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
])
clf.fit(X_train, y_train_cls)
y_pred_cls = clf.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test_cls, y_pred_cls):.3f}\n")
print(classification_report(y_test_cls, y_pred_cls, labels=labels))

ConfusionMatrixDisplay.from_predictions(y_test_cls, y_pred_cls, labels=labels, cmap="Blues")
plt.title("Матрица ошибок")
plt.show()

**Матрица ошибок:** по диагонали стоят правильные ответы. Обычно модель путает только соседние классы (дешёвый ↔ средний, средний ↔ дорогой) и почти никогда не путает «дешёвый» с «дорогим».

## Итоги

| Шаг | Что сделали |
|---|---|
| EDA | нашли скошенность цены → обучаемся на log; нашли выбросы → удалили; выявили ключевые признаки |
| Препроцессинг | `Pipeline` + `ColumnTransformer`: импутация, масштабирование, one-hot, без утечки данных |
| Baseline | точка отсчёта, медианная цена |
| Модели | сравнили 4 модели через 5-fold CV |
| Тюнинг | `GridSearchCV` для GradientBoosting |
| Оценка | MAE / RMSE / R² на отложенном тесте, графики остатков |
| Интерпретация | permutation importance: качество и площадь важнее всего |
| Деплой | сохранили пайплайн в `.pkl`, предсказали цену нового дома |

**Что можно улучшить:**
- добавить больше признаков из 80 доступных (например, `TotalSF` = подвал + 1 этаж + 2 этаж);
- попробовать `HistGradientBoostingRegressor`, XGBoost или LightGBM;
- использовать `RandomizedSearchCV` для более широкого поиска гиперпараметров;
- для своих данных (например, Дублин) достаточно заменить шаг 2 загрузкой своего CSV и списки `num_features` / `cat_features`.